In [13]:
from pydantic import Field, SecretStr, ValidationError, model_validator
from pydantic_settings import BaseSettings, SettingsConfigDict
import httpx

/Users/gasparcartasso/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/typing.py:1123: RuntimeWarning: coroutine 'OllamaProvider.agenerate' was never awaited
  def _generic_init_subclass(cls, *args, **kwargs):


In [2]:
import asyncio
import time
from dataclasses import asdict, dataclass
from typing import AsyncIterator, Protocol
@dataclass(frozen=True)
class GenerationResult:
    text: str
    model: str
    provider: str
    latency_ms: float
    input_tokens: int
    output_tokens: int
    finish_reason: str = "stop"
    def to_dict(self) -> dict:
        return asdict(self)
class LLMProvider(Protocol):
    async def agenerate(
        self,
        prompt: str,
        *,
        system: str | None = None,
        temperature: float = 0.2,
    ) -> GenerationResult: ...
    async def astream(
        self,
        prompt: str,
        *,
        system: str | None = None,
        temperature: float = 0.2,
    ) -> AsyncIterator[str]: ...

In [30]:
class OllamaProvider:
    def __init__(
        self,
        model: str = "qwen3.5:4b",
        base_url: str = "http://localhost:11434",
        timeout_seconds: float = 60.0,
        fallback=None,
    ):
        self.model = model
        self.base_url = base_url.rstrip("/")
        self.timeout_seconds = timeout_seconds
        self.fallback = fallback

    async def agenerate(self, prompt: str, *, system=None, temperature=0.2):
        if not prompt.strip():
            raise ValueError("El prompt no puede estar vacío")

        payload = {
            "model": self.model,
            "prompt": prompt,
            "system": system or "",
            "stream": False,
            "options": {"temperature": temperature},
        }
        started = time.perf_counter()
        try:
            async with httpx.AsyncClient(timeout=self.timeout_seconds) as client:
                response = await client.post(f"{self.base_url}/api/generate", json=payload)
                response.raise_for_status()
                data = response.json()
        except (httpx.HTTPError, ValueError) as exc:
            if self.fallback is not None:
                return await self.fallback.agenerate(
                    prompt, system=system, temperature=temperature
                )
            raise LocalProviderError(f"Ollama no disponible: {exc}") from exc

        text = str(data.get("response"))
        if not text:
            raise LocalProviderError("Ollama devolvió una respuesta vacía")

        return GenerationResult(
            text=text,
            model=self.model,
            provider="ollama",
            latency_ms=round((time.perf_counter() - started) * 1000, 2),
            input_tokens=int(data.get("prompt_eval_count", 0) or 0),
            output_tokens=int(data.get("eval_count", 0) or 0),
        )


In [17]:
import httpx
from httpx import TimeoutException

async def generate():
    try:
        async with httpx.AsyncClient(timeout=30.0) as client:  # timeout mayor
            response = await client.post(
                "http://localhost:11434/api/generate",
                json={"model": "qwen3.5:4b", "prompt": "Hola", "stream": False}
            )
            return response.json()
    except TimeoutException:
        print("Ollama no respondió. Verificá que el servidor esté corriendo y el modelo pullado.")
    except Exception as e:
        print(f"Error de conexión: {e}")
    return None

In [25]:
response = await generate()

In [29]:
response['response']

'¡Hola! 👋 ¿Cómo estás? \n\n¿Hay algo en lo que pueda ayudarte hoy, ya sea responder preguntas, crear contenido o simplemente tener una conversación interesante?'

In [31]:
await OllamaProvider().agenerate(prompt="Hola, ¿cómo estás?", system="Eres un asistente útil.", temperature=0.5)

GenerationResult(text='¡Hola! Estoy muy bien, gracias por preguntar. ¡Espero que tú también estés teniendo un gran día! \n\n¿En qué puedo ayudarte hoy?', model='qwen3.5:4b', provider='ollama', latency_ms=31267.95, input_tokens=28, output_tokens=754, finish_reason='stop')